# 📊 Customer Churn Prediction,  XGBoost + SHAP Explainability

**Author:** Muhammad Zohaib | [GitHub](https://github.com/Zohaib-mzb)  
**Live Dashboard:** [Open on Streamlit](https://customerchurndashboardai.streamlit.app/)  
**Dataset:** IBM Telco Customer Churn (7,043 customers, 21 features)

## What this notebook covers

This is a complete, business-focused machine learning project on 
customer churn prediction — from raw data to a deployed dashboard.

**Phase 1 — Data Understanding**
Loading the dataset, identifying data types, fixing the hidden 
TotalCharges issue, checking class imbalance.

**Phase 2 — Exploratory Data Analysis**
8 charts answering specific business questions — churn by contract 
type, tenure, monthly charges, payment method, internet service, 
and support services. Each chart followed by a plain-English 
business insight.

**Phase 3 — Data Preprocessing**
Binary encoding, one-hot encoding, stratified train/test split, 
StandardScaler for numerical features. Full documentation of why 
each decision was made.

**Phase 4 — Model Training and Evaluation**
Four models trained and compared: Logistic Regression, Decision Tree, 
Random Forest, XGBoost. ROC curves, confusion matrix, and full 
metric comparison table. XGBoost selected as the production model.

**Phase 5 — SHAP Explainability**
SHAP TreeExplainer showing which features drive each prediction. 
Global feature importance + individual customer explanation.

---

## Key Finding

Contract type alone explains most of the churn difference. 
Month-to-month customers churn at ~43% vs ~11% for two-year 
contract holders. The single highest-ROI retention strategy: 
incentivise customers to commit to longer contracts.

---

## Live Dashboard

The trained model is deployed as an interactive Streamlit dashboard 
where you can explore segment-level churn rates and predict churn 
probability for any individual customer with SHAP explanation.

🔗 [Open Dashboard](https://customerchurndashboardai.streamlit.app/)  
📁 [GitHub Repo](https://github.com/Zohaib-mzb/Customer-Churn-Prediction-Dashboard)

In [ ]:
!pip install xgboost shap --quiet
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, ConfusionMatrixDisplay)
import xgboost as xgb
import shap

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('Set2')

print("All libraries loaded successfully.")

In [ ]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## Phase 1: Data Understanding
First look at the structure, size, column types, and basic statistics.

In [ ]:
print("DATASET SHAPE:\n")
print(f"Rows (customers): {df.shape[0]:,}")
print(f"Columns (features): {df.shape[1]}")
print("\n")
print("COLUMN NAMES AND DATA TYPES")
print("\n")
print(df.dtypes)

Missing values checking

In [ ]:
print("MISSING VALUES CHECK \n")


missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})
missing_df = missing_df[missing_df['Missing Count'] > 0]

if missing_df.empty:
    print("No missing values found at surface level.")
    print("NOTE: TotalCharges may have hidden blank strings — we will check that next.")
else:
    print(missing_df)

Fixing the hidden TotalCharges issue

In [ ]:
print("TotalCharges column dtype:", df['TotalCharges'].dtype)
print("\nSample values in TotalCharges:")
print(df['TotalCharges'].head(10).tolist())

blank_count = df[df['TotalCharges'].str.strip() == '']['TotalCharges'].count()
print(f"\nBlank TotalCharges entries found: {blank_count}")

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

print(f"\nAfter fix — TotalCharges dtype: {df['TotalCharges'].dtype}")
print("Missing values in TotalCharges:", df['TotalCharges'].isnull().sum())
print("\nData issue resolved and documented.")

Statistical summary

In [ ]:
print("NUMERICAL COLUMNS — STATISTICAL SUMMARY")
print("-" * 60)
print(df.describe())

Target variable overview

In [ ]:
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100


print("TARGET VARIABLE — CHURN DISTRIBUTION \n")
print(f"Customers who stayed:  {churn_counts['No']:,}  ({churn_pct['No']:.1f}%)")
print(f"Customers who churned: {churn_counts['Yes']:,}  ({churn_pct['Yes']:.1f}%)")
print(f"\nChurn Rate: {churn_pct['Yes']:.1f}%")
print("\nBUSINESS NOTE:")
print("This is an imbalanced dataset — only ~26% of customers churned.")
print("A naive model predicting 'No Churn' every time would be 74% accurate.")
print("This is why we must use precision, recall, F1, and ROC-AUC — not just accuracy.")

Unique values in categorical columns

In [ ]:
categorical_cols = df.select_dtypes(include='object').columns.tolist()
categorical_cols = [c for c in categorical_cols if c != 'customerID']

print("CATEGORICAL COLUMNS — UNIQUE VALUES")
for col in categorical_cols:
    unique_vals = df[col].unique().tolist()
    print(f"\n{col} ({len(unique_vals)} unique values):")
    print(f"  {unique_vals}")

## Phase 1 Key Observations

1. **7,043 customers**, 21 features clean, structured dataset with no
   major missing values.
2. **TotalCharges had a hidden data quality issue** : stored as string with
   blank spaces for new customers. Fixed by converting to numeric and
   filling blanks with 0.
3. **Churn rate is ~26.5%** : this is a class imbalance problem. Standard
   accuracy is not a reliable metric here. We will use F1-score and
   ROC-AUC as primary evaluation metrics.
4. **Three numerical features:** tenure, MonthlyCharges, TotalCharges.
5. **Fifteen binary categorical features** (Yes/No) plus gender, contract
   type, payment method, and internet service type.
6. **No duplicate rows detected** : data is clean and ready for EDA.

## Phase 2 : Exploratory Data Analysis
Finding the business story inside the data. Each chart below answers
a specific business question and is followed by a plain English insight.

Overall churn donut chart

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

colors = ['#2ecc71', '#e74c3c']
sizes = [churn_counts['No'], churn_counts['Yes']]
labels = [f"Stayed\n{churn_counts['No']:,} ({churn_pct['No']:.1f}%)",
          f"Churned\n{churn_counts['Yes']:,} ({churn_pct['Yes']:.1f}%)"]

wedges, texts = ax.pie(sizes, labels=labels, colors=colors,
                        startangle=90, wedgeprops=dict(width=0.5))

ax.set_title('Overall Customer Churn Rate', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('chart1_churn_overview.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nBUSINESS INSIGHT:")
print("1 in 4 customers is leaving. At scale, this represents significant")
print("lost revenue. Even reducing churn by 5% could have a major financial impact.")

Churn by contract type

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

contract_churn = df.groupby('Contract')['Churn'].apply(
    lambda x: (x == 'Yes').sum() / len(x) * 100
).reset_index()
contract_churn.columns = ['Contract', 'ChurnRate']
contract_churn = contract_churn.sort_values('ChurnRate', ascending=False)

bars = ax.bar(contract_churn['Contract'], contract_churn['ChurnRate'],
              color=['#e74c3c', '#f39c12', '#2ecc71'], width=0.5, edgecolor='white')

for bar, val in zip(bars, contract_churn['ChurnRate']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=12)

ax.set_title('Churn Rate by Contract Type', fontsize=15, fontweight='bold')
ax.set_xlabel('Contract Type', fontsize=12)
ax.set_ylabel('Churn Rate (%)', fontsize=12)
ax.set_ylim(0, 55)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.savefig('chart2_contract_churn.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nBUSINESS INSIGHT:")
print("Contract type is the single strongest predictor of churn.")
print(f"Month-to-month customers churn at {contract_churn.iloc[0]['ChurnRate']:.1f}% ")
print("compared to much lower rates for one and two-year contract holders.")
print("The clearest retention strategy: incentivise customers to move")
print("from month-to-month to longer contracts.")

Churn by tenure groups

In [ ]:
# Create tenure groups
df['TenureGroup'] = pd.cut(df['tenure'],
                            bins=[0, 12, 24, 48, 72],
                            labels=['0-12 months', '13-24 months',
                                    '25-48 months', '49-72 months'])

fig, ax = plt.subplots(figsize=(10, 5))

tenure_churn = df.groupby('TenureGroup', observed=True)['Churn'].apply(
    lambda x: (x == 'Yes').sum() / len(x) * 100
).reset_index()
tenure_churn.columns = ['TenureGroup', 'ChurnRate']

bars = ax.bar(tenure_churn['TenureGroup'].astype(str),
              tenure_churn['ChurnRate'],
              color=['#e74c3c', '#e67e22', '#3498db', '#2ecc71'],
              width=0.5, edgecolor='white')

for bar, val in zip(bars, tenure_churn['ChurnRate']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=12)

ax.set_title('Churn Rate by Customer Tenure', fontsize=15, fontweight='bold')
ax.set_xlabel('Tenure Group', fontsize=12)
ax.set_ylabel('Churn Rate (%)', fontsize=12)
ax.set_ylim(0, 60)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.savefig('chart3_tenure_churn.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nBUSINESS INSIGHT:")
print("New customers (0-12 months) are by far the highest churn risk.")
print("If a customer survives past 2 years, they become dramatically more loyal.")
print("This suggests the critical retention window is the first 12 months.")
print("Early engagement programs and onboarding quality matter most.")

Monthly charges vs churn

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

churned = df[df['Churn'] == 'Yes']['MonthlyCharges']
stayed = df[df['Churn'] == 'No']['MonthlyCharges']

ax.hist(stayed, bins=40, alpha=0.6, color='#2ecc71', label=f'Stayed (avg: ${stayed.mean():.0f}/mo)')
ax.hist(churned, bins=40, alpha=0.6, color='#e74c3c', label=f'Churned (avg: ${churned.mean():.0f}/mo)')

ax.axvline(stayed.mean(), color='#27ae60', linestyle='--', linewidth=2)
ax.axvline(churned.mean(), color='#c0392b', linestyle='--', linewidth=2)

ax.set_title('Monthly Charges Distribution — Churned vs Stayed', fontsize=15, fontweight='bold')
ax.set_xlabel('Monthly Charges ($)', fontsize=12)
ax.set_ylabel('Number of Customers', fontsize=12)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('chart4_monthly_charges.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nBUSINESS INSIGHT:")
print(f"Customers who churned paid on average ${churned.mean():.0f}/month,")
print(f"vs ${stayed.mean():.0f}/month for those who stayed.")
print("Higher paying customers are more likely to leave possibly due to perceived value for money. Price sensitivity is a real churn driver.")


Churn by internet service

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

internet_churn = df.groupby('InternetService')['Churn'].apply(
    lambda x: (x == 'Yes').sum() / len(x) * 100
).reset_index()
internet_churn.columns = ['InternetService', 'ChurnRate']

bars = ax.bar(internet_churn['InternetService'],
              internet_churn['ChurnRate'],
              color=['#3498db', '#e74c3c', '#2ecc71'],
              width=0.5, edgecolor='white')

for bar, val in zip(bars, internet_churn['ChurnRate']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=12)

ax.set_title('Churn Rate by Internet Service Type', fontsize=15, fontweight='bold')
ax.set_xlabel('Internet Service', fontsize=12)
ax.set_ylabel('Churn Rate (%)', fontsize=12)
ax.set_ylim(0, 55)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.savefig('chart5_internet_churn.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nBUSINESS INSIGHT:")
print("Fibre optic customers churn at a much higher rate than DSL customers.")
print("This is counterintuitive — faster internet should mean happier customers.")
print("Possible reasons: higher price point, higher expectations, stronger competition.")
print("This segment needs targeted retention campaigns.")

Churn by payment method

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

payment_churn = df.groupby('PaymentMethod')['Churn'].apply(
    lambda x: (x == 'Yes').sum() / len(x) * 100
).reset_index()
payment_churn.columns = ['PaymentMethod', 'ChurnRate']
payment_churn = payment_churn.sort_values('ChurnRate', ascending=False)

bars = ax.barh(payment_churn['PaymentMethod'],
               payment_churn['ChurnRate'],
               color=['#e74c3c', '#e67e22', '#3498db', '#2ecc71'],
               edgecolor='white', height=0.5)

for bar, val in zip(bars, payment_churn['ChurnRate']):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontweight='bold', fontsize=12)

ax.set_title('Churn Rate by Payment Method', fontsize=15, fontweight='bold')
ax.set_xlabel('Churn Rate (%)', fontsize=12)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.savefig('chart6_payment_churn.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nBUSINESS INSIGHT:")
print("Electronic check users churn at almost double the rate of customers who use automatic payment methods.Customers on automatic payments are more committed and passive,\nthey don't actively think about cancelling every month.")
print("Strategy: offer discounts to migrate electronic check users to auto-pay.")

Support services vs churn

In [ ]:
support_cols = ['TechSupport', 'OnlineSecurity', 'OnlineBackup']
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, col in enumerate(support_cols):
    churn_data = df.groupby(col)['Churn'].apply(
        lambda x: (x == 'Yes').sum() / len(x) * 100
    ).reset_index()
    churn_data.columns = [col, 'ChurnRate']
    churn_data = churn_data[churn_data[col] != 'No internet service']

    colors_map = {'No': '#e74c3c', 'Yes': '#2ecc71'}
    bar_colors = [colors_map.get(v, '#3498db') for v in churn_data[col]]

    bars = axes[i].bar(churn_data[col], churn_data['ChurnRate'],
                       color=bar_colors, width=0.4, edgecolor='white')

    for bar, val in zip(bars, churn_data['ChurnRate']):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                     f'{val:.1f}%', ha='center', va='bottom',
                     fontweight='bold', fontsize=12)

    axes[i].set_title(col, fontsize=13, fontweight='bold')
    axes[i].set_ylabel('Churn Rate (%)' if i == 0 else '')
    axes[i].set_ylim(0, 55)
    axes[i].yaxis.set_major_formatter(mtick.PercentFormatter())

fig.suptitle('Support Services vs Churn Rate', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('chart7_support_churn.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nBUSINESS INSIGHT:")
print("Customers WITHOUT tech support, online security, or online backup")
print("churn at roughly double the rate of those WITH these services.")
print("These add-on services are not just revenue streams — they are")
print("retention tools. Upselling them actively reduces churn risk.")

Correlation heatmap

In [ ]:
df_corr = df.copy()
df_corr['Churn_binary'] = (df_corr['Churn'] == 'Yes').astype(int)

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn_binary']
corr_matrix = df_corr[num_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, ax=ax,
            linewidths=0.5, cbar_kws={"shrink": 0.8})

ax.set_title('Correlation Matrix — Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('chart8_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nBUSINESS INSIGHT:")
print("Tenure has a NEGATIVE correlation with churn (-0.35) —")
print("longer-tenured customers are more loyal, confirming earlier finding.")
print("MonthlyCharges has a slight POSITIVE correlation (+0.19) —")
print("higher bills slightly increase churn risk.")
print("TotalCharges is highly correlated with tenure (0.83) — not surprising,")
print("longer customers naturally accumulate higher total spend.")

## Phase 2 : EDA Summary: Key Business Findings

| Finding | Churn Rate | Business Implication |
|---|---|---|
| Month-to-month contract | ~43% | Strongest churn predictor — incentivise long contracts |
| 0–12 months tenure | ~48% | Critical retention window — improve early onboarding |
| No tech support | ~41% | Upsell support services as retention tools |
| Electronic check payment | ~45% | Migrate to auto-pay with discounts |
| Fibre optic internet | ~42% | High expectations — improve service quality |
| High monthly charges | Higher risk | Review pricing for value perception |

**Top 3 actionable strategies from EDA alone:**
1. Lock customers into annual/two-year contracts with incentives
2. Intensive first-year engagement programme for new customers
3. Auto-pay migration campaign for electronic check users

## Phase 3 : Data Preprocessing and Feature Engineering
Preparing the raw data for ML models. Encoding categorical variables,
handling class imbalance, and creating the final feature matrix.

In [ ]:
df_ml = df.copy()
df_ml = df_ml.drop(columns=['customerID'])

print("customerID column dropped.")
print(f"Remaining columns: {df_ml.shape[1]}")

Encode binary Yes/No columns

In [ ]:
# Columns with simple Yes/No values — convert to 1/0
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling',
               'Churn', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
               'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

# For columns that have Yes/No/No internet service — map Yes=1, No=0, No internet service=0
for col in binary_cols:
    if col in df_ml.columns:
        df_ml[col] = df_ml[col].map({'Yes': 1, 'No': 0,
                                      'No internet service': 0,
                                      'No phone service': 0})

print("Binary encoding complete.")
print(df_ml[binary_cols[:6]].head())

Encode gender

In [ ]:
df_ml['gender'] = df_ml['gender'].map({'Male': 1, 'Female': 0})
print("Gender encoded: Male=1, Female=0")

One hot encode multi-category columns

In [ ]:
multi_cat_cols = ['InternetService', 'Contract', 'PaymentMethod', 'TenureGroup']

df_ml = pd.get_dummies(df_ml, columns=multi_cat_cols, drop_first=False)

bool_cols = df_ml.select_dtypes(include='bool').columns
df_ml[bool_cols] = df_ml[bool_cols].astype(int)

print("One-hot encoding complete.")
print(f"Total columns after encoding: {df_ml.shape[1]}")
print("\nNew columns created:")
new_cols = [c for c in df_ml.columns if any(prefix in c for prefix in multi_cat_cols)]
for c in new_cols:
    print(f"  {c}")

Verify no remaining object columns

In [ ]:
remaining_objects = df_ml.select_dtypes(include='object').columns.tolist()

if remaining_objects:
    print(f"WARNING — These columns are still strings: {remaining_objects}")
    print("They need to be encoded before modelling.")
else:
    print("All columns are now numeric. Ready for modelling.")

print(f"\nDataset shape: {df_ml.shape}")
print("\nSample of processed data:")
df_ml.head(3)

Feature/target split

In [ ]:
X = df_ml.drop(columns=['Churn'])
y = df_ml['Churn']

print(f"Features (X): {X.shape[1]} columns, {X.shape[0]:,} rows")
print(f"Target (y): {y.shape[0]:,} rows")
print(f"\nTarget distribution:")
print(f"  Churn = 0 (Stayed):  {(y == 0).sum():,} ({(y == 0).mean()*100:.1f}%)")
print(f"  Churn = 1 (Churned): {(y == 1).sum():,} ({(y == 1).mean()*100:.1f}%)")

Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 80% train, 20% test
    random_state=42,     # reproducible results
    stratify=y           # maintain churn ratio in both splits
)

print("Train/Test Split Complete")
print("=" * 40)
print(f"Training set:  {X_train.shape[0]:,} rows ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Test set:      {X_test.shape[0]:,} rows ({X_test.shape[0]/len(X)*100:.0f}%)")
print(f"\nTraining churn rate: {y_train.mean()*100:.1f}%")
print(f"Test churn rate:     {y_test.mean()*100:.1f}%")
print("\nStratification worked — churn rate is preserved in both splits.")

Feature scaling

In [ ]:
# Scale numerical features — important for Logistic Regression
# Tree-based models (Random Forest, XGBoost) don't strictly need scaling
# but it doesn't hurt and keeps the pipeline consistent

scaler = StandardScaler()

# Only scale numerical columns
num_features = ['tenure', 'MonthlyCharges', 'TotalCharges']

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[num_features] = scaler.fit_transform(X_train[num_features])
X_test_scaled[num_features] = scaler.transform(X_test[num_features])

print("Feature scaling complete.")
print(f"Scaled features: {num_features}")
print("\nBefore scaling (MonthlyCharges):")
print(f"  Mean: {X_train['MonthlyCharges'].mean():.2f}, Std: {X_train['MonthlyCharges'].std():.2f}")
print("\nAfter scaling (MonthlyCharges):")
print(f"  Mean: {X_train_scaled['MonthlyCharges'].mean():.4f}, Std: {X_train_scaled['MonthlyCharges'].std():.4f}")

In [ ]:
# Save cleaned data for dashboard use later
X_train_scaled.to_csv('X_train.csv', index=False)
X_test_scaled.to_csv('X_test.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)
df_ml.to_csv('df_processed.csv', index=False)

print("Processed datasets saved:")
print("  X_train.csv, X_test.csv, y_train.csv, y_test.csv, df_processed.csv")


## Phase 3 : Preprocessing Summary

| Step | Action | Why |
|---|---|---|
| Dropped customerID | Not predictive | Random unique ID adds noise |
| Binary encoding | Yes=1, No=0 | ML models need numbers not strings |
| One-hot encoding | Contract, InternetService, PaymentMethod | Multi-category columns need dummy variables |
| Stratified split | 80/20 with stratify=y | Preserves churn ratio in both train/test sets |
| StandardScaler | Scaled tenure, MonthlyCharges, TotalCharges | Normalises range for Logistic Regression |


## Phase 4 : Model Training and Evaluation
Training four ML models, comparing them honestly, selecting the best
performer, and adding SHAP explainability to understand WHY the model
makes each prediction.

In [ ]:
# Reusable function — evaluates any model and returns a clean results dict
def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    y_pred  = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]

    results = {
        'Model':     name,
        'Accuracy':  round(accuracy_score(y_te, y_pred)   * 100, 2),
        'Precision': round(precision_score(y_te, y_pred)  * 100, 2),
        'Recall':    round(recall_score(y_te, y_pred)     * 100, 2),
        'F1-Score':  round(f1_score(y_te, y_pred)         * 100, 2),
        'ROC-AUC':   round(roc_auc_score(y_te, y_proba)   * 100, 2),
    }

    print(f"\n{'='*45}")
    print(f"  {name}")
    print(f"{'='*45}")
    for metric, val in results.items():
        if metric != 'Model':
            bar = '█' * int(val / 5) + '░' * (20 - int(val / 5))
            print(f"  {metric:<12} [{bar}] {val}%")

    return results, model, y_pred, y_proba

In [ ]:
# Fix: TenureGroup column is still present in saved CSVs
# This happens when the drop cell ran but the CSVs were already saved before it
# One-line fix — drop it from all four dataframes

drop_cols = ['TenureGroup', 'ChargesGroup']

for name, dataframe in [('X_train', X_train), ('X_test', X_test)]:
    for col in drop_cols:
        if col in dataframe.columns:
            dataframe.drop(columns=[col], inplace=True)
            print(f"Dropped '{col}' from {name}")

# Also fix any remaining object columns
obj_cols_train = X_train.select_dtypes(include='object').columns.tolist()
obj_cols_test  = X_test.select_dtypes(include='object').columns.tolist()

if obj_cols_train:
    print(f"\nWARNING — Object columns still in X_train: {obj_cols_train}")
    X_train.drop(columns=obj_cols_train, inplace=True)
    print("Dropped from X_train.")

if obj_cols_test:
    print(f"WARNING — Object columns still in X_test: {obj_cols_test}")
    X_test.drop(columns=obj_cols_test, inplace=True)
    print("Dropped from X_test.")

# Verify everything is numeric now
remaining = X_train.select_dtypes(include='object').columns.tolist()
if not remaining:
    print("\nAll columns are numeric. Ready to train.")
else:
    print(f"\nStill has string columns: {remaining}")

print(f"\nX_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")

# Overwrite the saved CSVs with the clean version
X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)
print("\nCSV files updated.")

Train models

In [ ]:
all_results = []

# ── Model 1: Logistic Regression ────────────────────────────────
res, lr_model, lr_pred, lr_proba = evaluate_model(
    'Logistic Regression',
    LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    X_train, X_test, y_train, y_test
)
all_results.append(res)

# ── Model 2: Decision Tree ───────────────────────────────────────
res, dt_model, dt_pred, dt_proba = evaluate_model(
    'Decision Tree',
    DecisionTreeClassifier(class_weight='balanced', max_depth=6, random_state=42),
    X_train, X_test, y_train, y_test
)
all_results.append(res)

# ── Model 3: Random Forest ───────────────────────────────────────
res, rf_model, rf_pred, rf_proba = evaluate_model(
    'Random Forest',
    RandomForestClassifier(class_weight='balanced', n_estimators=200,
                           max_depth=10, random_state=42, n_jobs=-1),
    X_train, X_test, y_train, y_test
)
all_results.append(res)

# ── Model 4: XGBoost ────────────────────────────────────────────
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()

res, xgb_model, xgb_pred, xgb_proba = evaluate_model(
    'XGBoost',
    xgb.XGBClassifier(scale_pos_weight=scale_pos, n_estimators=300,
                      max_depth=6, learning_rate=0.05,
                      subsample=0.8, colsample_bytree=0.8,
                      use_label_encoder=False, eval_metric='logloss',
                      random_state=42, n_jobs=-1),
    X_train, X_test, y_train, y_test
)
all_results.append(res)

Model comparison table

In [ ]:
results_df = pd.DataFrame(all_results).set_index('Model')

print("\n" + "="*65)
print("  MODEL COMPARISON TABLE")
print("="*65)
print(results_df.to_string())
print("="*65)
print(f"\nBest F1-Score:  {results_df['F1-Score'].idxmax()} "
      f"({results_df['F1-Score'].max()}%)")
print(f"Best ROC-AUC:   {results_df['ROC-AUC'].idxmax()} "
      f"({results_df['ROC-AUC'].max()}%)")

Model comparison bar chart

In [ ]:
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
x      = np.arange(len(metrics))
width  = 0.2
colors = ['#3498db', '#e67e22', '#2ecc71', '#9b59b6']

fig, ax = plt.subplots(figsize=(14, 6))

for i, (model_name, row) in enumerate(results_df.iterrows()):
    bars = ax.bar(x + i * width, row[metrics], width,
                  label=model_name, color=colors[i],
                  alpha=0.85, edgecolor='white')

ax.set_title('Model Comparison — All Metrics', fontsize=15, fontweight='bold')
ax.set_xlabel('Metric', fontsize=12)
ax.set_ylabel('Score (%)', fontsize=12)
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylim(50, 100)
ax.legend(fontsize=11, loc='lower right')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.savefig('chart9_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nBUSINESS NOTE:")
print("For churn prediction, Recall is the most important metric.")
print("Missing a churner (false negative) costs more than a false alarm.")
print("A customer you incorrectly flag can be given a retention offer —")
print("but a churner you miss is gone forever.")

ROC Curve Comparison

In [ ]:
best_model      = xgb_model
best_pred       = xgb_pred
best_model_name = 'XGBoost'

fig, ax = plt.subplots(figsize=(7, 6))

cm = confusion_matrix(y_test, best_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Stayed', 'Churned'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')

ax.set_title(f'Confusion Matrix — {best_model_name}',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('chart11_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"\nConfusion Matrix Breakdown:")
print(f"  True Negatives  (correctly predicted Stayed):  {tn:,}")
print(f"  False Positives (predicted Churn, actually Stayed): {fp:,}")
print(f"  False Negatives (predicted Stayed, actually Churned): {fn:,}")
print(f"  True Positives  (correctly predicted Churn):  {tp:,}")
print(f"\nOf {tp + fn:,} actual churners in test set, model caught {tp:,} ({tp/(tp+fn)*100:.1f}%)")
print(f"False alarm rate: {fp:,} customers flagged incorrectly ({fp/(fp+tn)*100:.1f}% of non-churners)")

SHAP explainability

In [ ]:
print("Calculating SHAP values — this may take 30-60 seconds...")

explainer   = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

print("SHAP values calculated.")

SHAP summary bar chart

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

shap.summary_plot(shap_values, X_test,
                  plot_type='bar',
                  max_display=15,
                  show=False)

plt.title(f'Top 15 Features — SHAP Importance ({best_model_name})',
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('chart12_shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nBUSINESS INSIGHT:")
print("SHAP shows the TRUE importance of each feature to the model's decisions.")
print("Unlike basic feature importance, SHAP accounts for feature interactions.")
print("The longer the bar, the more that feature influences churn predictions.")

beeswarm plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

shap.summary_plot(shap_values, X_test,
                  max_display=15,
                  show=False)

plt.title(f'SHAP Value Distribution ({best_model_name})',
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('chart13_shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nHOW TO READ THIS CHART:")
print("Each dot = one customer in the test set.")
print("Red dots = high feature value, Blue dots = low feature value.")
print("Dots on the RIGHT = pushed the prediction TOWARD churn.")
print("Dots on the LEFT  = pushed the prediction AWAY from churn.")
print("\nExample reading: 'Month-to-month contract (red dot, right side)'")
print("means customers ON that contract are pushed strongly toward churn.")

In [ ]:
import pickle

with open('best_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

with open('scaler.pkl', 'wb') as f:
    from sklearn.preprocessing import StandardScaler
    # Save feature names too for dashboard use
    pickle.dump({
        'feature_names': X_train.columns.tolist(),
        'num_features': ['tenure', 'MonthlyCharges', 'TotalCharges']
    }, f)

print("Model saved as best_model.pkl")
print("Feature info saved as scaler.pkl")
print("\nThese files will be loaded by the Streamlit dashboard.")

## Phase 4 : Model Results Summary

| Model | Accuracy | Precision | Recall | F1-Score | ROC-AUC |
|---|---|---|---|---|---|
| Logistic Regression | ~80% | ~65% | ~76% | ~70% | ~84% |
| Decision Tree | ~78% | ~60% | ~72% | ~65% | ~77% |
| Random Forest | ~82% | ~70% | ~74% | ~72% | ~86% |
| **XGBoost** | **~82%** | **~68%** | **~78%** | **~73%** | **~87%** |

**Winner: XGBoost** — highest ROC-AUC and best balance of Precision/Recall.

**Why XGBoost wins:**
- Handles class imbalance well via scale_pos_weight
- Captures complex non-linear relationships between features
- Gradient boosting builds trees sequentially, each correcting the last
- Consistently outperforms on tabular business data

**Top SHAP features driving churn:**
1. Contract type (month-to-month = strong churn signal)
2. Tenure (short tenure = high risk)
3. Monthly charges (higher bills = higher risk)
4. Internet service type (Fibre optic = higher risk)
5. Tech support / Online security (absence = higher risk)



In [ ]:
!pip install streamlit pyngrok --quiet
print("Streamlit installed.")